# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

from tqdm.auto import tqdm
import joblib
import torch

sys.path.append("../../")


from src.evaluation.perf_eval import display_pred_perf
from src.display.display_df import df_to_latex
from src.evaluation.consolidate_across_data import \
    all_data_perf, process_tabular_pred_perf, process_image_pred_perf, \
    process_tabular_ue_perf, process_image_ue_perf, process_image_ood_perf, \
    all_data_perf_improvement
from src.evaluation.ue_metrics import display_ue_perf, display_ood_perf

tabular_data_names = ["schs-lung", "schs-crc"]
image_data_names = ["octmnist", "blood"]
data_names = tabular_data_names+image_data_names

# Prediction Perf

## In-Distribution

In [ ]:
all_pred_perf_df = all_data_perf(
    data_names, columns=["AUC"],  # , "Acc"
    process_df_funcs=[
        process_tabular_pred_perf, process_tabular_pred_perf, 
        process_image_pred_perf, process_image_pred_perf],
    order = ["egRUE & RUE", "MCD", "DE", "PN", "GPC", "DEC", "BNN"]
)
display_pred_perf(all_pred_perf_df, consolidated=True)

In [ ]:
column_format_dict = {col: "max" for col in all_pred_perf_df.columns}
print(df_to_latex(all_pred_perf_df, column_format_dict=column_format_dict))

## Out-Distribution

In [ ]:
all_pred_perf_df_out = all_data_perf(
    data_names[2:], columns=["AUC"],  #, "Acc"
    process_df_funcs=[
        process_image_pred_perf, process_image_pred_perf],
    order = ["egRUE & RUE", "MCD", "DE", "PN", "DEC", "BNN"],
    pref_file="pred_perf_out.csv"
)
display_pred_perf(all_pred_perf_df_out, consolidated=True)

In [ ]:
column_format_dict = {col: "max" for col in all_pred_perf_df_out.columns}
print(df_to_latex(all_pred_perf_df_out, column_format_dict=column_format_dict))

# Uncertainty Estimation Perf

## Main UE Table

In [ ]:
all_ue_perf_df_tabular = all_data_perf(
    tabular_data_names, pref_file="ue_perf.csv",
    order=[
        "egRUE", #"RUE",
        "Entropy", "MCD", "DE", 
        "PN Alea", "PN Epis", "GPC"
    ],
    columns=["Pearson's Correlation", "AUROC", "AURC (0/1 Loss)", "Sigma-Risk Score (0.1)"],
    process_df_funcs=[process_tabular_ue_perf, process_tabular_ue_perf]
)
display_ue_perf(all_ue_perf_df_tabular, consolidated=True)
high_cols = ["Pearson's Correlation", "AUROC"]
low_cols = ["AURC (0/1 Loss)","Sigma-Risk Score (0.1)"]
column_format_dict = {col:"max" for col in all_ue_perf_df_tabular if col[1] in high_cols}
column_format_dict.update({col:"min" for col in all_ue_perf_df_tabular if col[1] in low_cols})
print(df_to_latex(all_ue_perf_df_tabular, column_format_dict=column_format_dict))

In [ ]:
all_ue_perf_df_image = all_data_perf(
    image_data_names, pref_file="ue_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    columns=["Pearson's Correlation", "AUROC", "AURC (0/1 Loss)", "Sigma-Risk Score (0.1)"],
    process_df_funcs=[process_image_ue_perf, process_image_ue_perf]
)
display_ue_perf(all_ue_perf_df_image, consolidated=True)
high_cols = ["Pearson's Correlation", "AUROC"]
low_cols = ["AURC (0/1 Loss)","Sigma-Risk Score (0.1)"]
column_format_dict = {col:"max" for col in all_ue_perf_df_image if col[1] in high_cols}
column_format_dict.update({col:"min" for col in all_ue_perf_df_image if col[1] in low_cols})
print(df_to_latex(all_ue_perf_df_image, column_format_dict=column_format_dict))

## Improvement Over Deep Ensemble

In [ ]:
all_data_perf_improvement(
    proposed_method="egRUE",
    data_names=data_names, pref_file="ue_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    columns=["Pearson's Correlation", "AUROC", "AURC (0/1 Loss)", "Sigma-Risk Score (0.1)"],
    process_df_funcs=[
        process_tabular_ue_perf, process_tabular_ue_perf, 
        process_image_ue_perf, process_image_ue_perf], sf=0
)

## Ablation

In [ ]:
all_ue_perf_df_image = all_data_perf(
    tabular_data_names, pref_file="ue_perf.csv",
    order=[
        "egRUE", "RUE",  
        # "Entropy", "MCD", "DE", 
        # "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    columns=["Pearson's Correlation", "AUROC", "AURC (0/1 Loss)", "Sigma-Risk Score (0.1)"],
    process_df_funcs=[process_image_ue_perf, process_image_ue_perf]
)
display_ue_perf(all_ue_perf_df_image, consolidated=True)
high_cols = ["Pearson's Correlation", "AUROC"]
low_cols = ["AURC (0/1 Loss)","Sigma-Risk Score (0.1)"]
column_format_dict = {col:"max" for col in all_ue_perf_df_image if col[1] in high_cols}
column_format_dict.update({col:"min" for col in all_ue_perf_df_image if col[1] in low_cols})
print(df_to_latex(all_ue_perf_df_image, column_format_dict=column_format_dict))

In [ ]:
all_ue_perf_df_image = all_data_perf(
    image_data_names, pref_file="ue_perf.csv",
    order=[
        "egRUE", "RUE",  
        # "Entropy", "MCD", "DE", 
        # "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    columns=["Pearson's Correlation", "AUROC", "AURC (0/1 Loss)", "Sigma-Risk Score (0.1)"],
    process_df_funcs=[process_image_ue_perf, process_image_ue_perf]
)
display_ue_perf(all_ue_perf_df_image, consolidated=True)
high_cols = ["Pearson's Correlation", "AUROC"]
low_cols = ["AURC (0/1 Loss)","Sigma-Risk Score (0.1)"]
column_format_dict = {col:"max" for col in all_ue_perf_df_image if col[1] in high_cols}
column_format_dict.update({col:"min" for col in all_ue_perf_df_image if col[1] in low_cols})
print(df_to_latex(all_ue_perf_df_image, column_format_dict=column_format_dict))

# OOD Detection Perf

## Main OOD Detection Table

In [ ]:
all_ood_perf_df_image = all_data_perf(
    image_data_names[:1], pref_file="ood_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Aleatoric", "PN Epistemic", "BNN", "DEC"
    ],
    process_df_funcs=[process_image_ood_perf],
)
display_ood_perf(all_ood_perf_df_image.iloc[:, [0,1,2]], consolidated=True)
column_format_dict = {col: "max" for col in all_ood_perf_df_image.iloc[:, [0,1,2]].columns}
print(df_to_latex(all_ood_perf_df_image.iloc[:, [0,1,2]], column_format_dict=column_format_dict))

In [ ]:
all_ood_perf_df_image = all_data_perf(
    image_data_names[1:], pref_file="ood_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Aleatoric", "PN Epistemic", "BNN", "DEC"
    ],
    process_df_funcs=[process_image_ood_perf],
)
display_ood_perf(all_ood_perf_df_image.iloc[:, [0,1,2]], consolidated=True)
column_format_dict = {col: "max" for col in all_ood_perf_df_image.iloc[:, [0,1,2]].columns}
print(df_to_latex(all_ood_perf_df_image.iloc[:, [0,1,2]], column_format_dict=column_format_dict))

## Improvement Over Deep Ensemble

In [ ]:
all_data_perf_improvement(
    proposed_method="egRUE",
    data_names=image_data_names[:1], pref_file="ood_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    column_direction=["max", "max", "max", "max"],
    process_df_funcs=[
        process_image_ue_perf, process_image_ue_perf], sf=0
)

In [ ]:
all_data_perf_improvement(
    proposed_method="egRUE",
    data_names=image_data_names[1:], pref_file="ood_perf.csv",
    order=[
        "egRUE", #"RUE",  
        "Entropy", "MCD", "DE", 
        "PN Alea", "PN Epis", "BNN", "DEC"
    ],
    column_direction=["max", "max", "max", "max"],
    process_df_funcs=[
        process_image_ue_perf, process_image_ue_perf], sf=0
)